# Half-hourly settlement data and panel (many-meter) operations

UK electricity settles in **half-hour periods**: 48 per day, 46 on the spring clock-change
day and 50 on the autumn one. Meter data arrives as `settlement_date` (a local date) plus
`settlement_period` (1..48) per meter, not as timestamps.

Every idea is shown first on a tiny frame (one or two meters, a handful of periods), then
once on the real 20-meter file.

**What's in here**
- what a settlement-period row looks like
- settlement period → UTC timestamp, step by step
- the clock-change days (46 and 50 periods)
- finding missing periods with a full grid
- wide vs long layouts
- `groupby` before `shift` / `rolling` / `diff` — the panel rule
- `transform`: a per-meter statistic on every row
- energy (sum) vs power (mean) when resampling
- bottom-up portfolio totals and the coincidence factor
- per-meter statistics in one `agg`
- a small panel model with honest per-meter features
- checklist

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)

## 1. What a settlement-period row looks like

One meter, one day, the first four periods. Period 1 is 00:00–00:30 **local** time.

In [2]:
toy = pd.DataFrame({
    "meter_id": ["M1", "M1", "M1", "M1"],
    "settlement_date": ["2023-07-01", "2023-07-01", "2023-07-01", "2023-07-01"],
    "settlement_period": [1, 2, 3, 4],
    "kwh": [0.20, 0.18, 0.15, 0.16],
})
toy

,meter_id,settlement_date,settlement_period,kwh
0,M1,2023-07-01,1,0.20
1,M1,2023-07-01,2,0.18
2,M1,2023-07-01,3,0.15
3,M1,2023-07-01,4,0.16


## 2. Settlement period → UTC timestamp, step by step

Step 1: parse the date. It is still a naive date (no timezone).

In [3]:
toy["date"] = pd.to_datetime(toy["settlement_date"])
toy[["settlement_date", "date"]]

,settlement_date,date
0,2023-07-01,2023-07-01
1,2023-07-01,2023-07-01
2,2023-07-01,2023-07-01
3,2023-07-01,2023-07-01


Step 2: say that this midnight is **London local** midnight.

In [4]:
toy["local_midnight"] = toy["date"].dt.tz_localize("Europe/London")
toy[["date", "local_midnight"]]

,date,local_midnight
0,2023-07-01,2023-07-01 00:00:00+01:00
1,2023-07-01,2023-07-01 00:00:00+01:00
2,2023-07-01,2023-07-01 00:00:00+01:00
3,2023-07-01,2023-07-01 00:00:00+01:00


The `+01:00` tells you July is British Summer Time: local midnight is 23:00 UTC the day before.

Step 3: convert that midnight to UTC.

In [5]:
toy["utc_midnight"] = toy["local_midnight"].dt.tz_convert("UTC")
toy[["local_midnight", "utc_midnight"]]

,local_midnight,utc_midnight
0,2023-07-01 00:00:00+01:00,2023-06-30 23:00:00+00:00
1,2023-07-01 00:00:00+01:00,2023-06-30 23:00:00+00:00
2,2023-07-01 00:00:00+01:00,2023-06-30 23:00:00+00:00
3,2023-07-01 00:00:00+01:00,2023-06-30 23:00:00+00:00


Step 4: add (period − 1) × 30 minutes **in UTC**.

In [6]:
toy["offset"] = pd.to_timedelta((toy["settlement_period"] - 1) * 30, unit="min")
toy["utc"] = toy["utc_midnight"] + toy["offset"]
toy[["settlement_period", "offset", "utc"]]

,settlement_period,offset,utc
0,1,0 days 00:00:00,2023-06-30 23:00:00+00:00
1,2,0 days 00:30:00,2023-06-30 23:30:00+00:00
2,3,0 days 01:00:00,2023-07-01 00:00:00+00:00
3,4,0 days 01:30:00,2023-07-01 00:30:00+00:00


Period 1 → 23:00 UTC (= 00:00 local), period 2 → 23:30 UTC, and so on.

Why not just add the offset to the local date and call it local time? Because on the
clock-change days some local times do not exist or exist twice.

## 3. The clock-change days

On 2023-03-26 clocks jump from 01:00 to 02:00: the day has 23 hours = 46 periods.
Take periods 1–4 of that day and build the timestamp the correct way.

In [7]:
dst = pd.DataFrame({"settlement_date": ["2023-03-26"] * 4, "settlement_period": [1, 2, 3, 4]})
utc_midnight = pd.to_datetime(dst["settlement_date"]).dt.tz_localize("Europe/London").dt.tz_convert("UTC")
dst["utc"] = utc_midnight + pd.to_timedelta((dst["settlement_period"] - 1) * 30, unit="min")
dst["local"] = dst["utc"].dt.tz_convert("Europe/London")
dst

,settlement_date,settlement_period,utc,local
0,2023-03-26,1,2023-03-26 00:00:00+00:00,2023-03-26 00:00:00+00:00
1,2023-03-26,2,2023-03-26 00:30:00+00:00,2023-03-26 00:30:00+00:00
2,2023-03-26,3,2023-03-26 01:00:00+00:00,2023-03-26 02:00:00+01:00
3,2023-03-26,4,2023-03-26 01:30:00+00:00,2023-03-26 02:30:00+01:00


Period 3 is 01:00 UTC, which is **02:00 local** — the hour 01:00–02:00 local never
happened. The UTC construction handles it without any special case.

The naive way (treat `date + offset` as local) breaks:

In [8]:
naive_local = pd.to_datetime(dst["settlement_date"]) + pd.to_timedelta((dst["settlement_period"] - 1) * 30, unit="min")
print(naive_local.tolist())
try:
    naive_local.dt.tz_localize("Europe/London")
except Exception as e:
    print(type(e).__name__, "->", str(e)[:70])

[Timestamp('2023-03-26 00:00:00'), Timestamp('2023-03-26 00:30:00'), Timestamp('2023-03-26 01:00:00'), Timestamp('2023-03-26 01:30:00')]
NonExistentTimeError -> 2023-03-26 01:00:00


Period 3 became 01:00 local, a time that does not exist. On the autumn day the same
approach gives ambiguous times (01:30 happens twice).

## 4. Load the real file

20 meters, all of 2023.

In [9]:
raw = pd.read_csv("../data/meter_halfhourly_2023.csv.gz")   # pandas reads gzip transparently
print(raw.shape)
raw.head()

(349439, 4)


,meter_id,settlement_date,settlement_period,kwh
0,M100012,2023-05-14,35,0.262
1,M100005,2023-06-05,25,0.117
2,M100013,2023-12-07,33,0.172
3,M100016,2023-08-21,4,0.304
4,M100014,2023-12-17,31,0.215


How many periods does each date have? Almost all 48; the two clock-change days differ.

In [10]:
periods_per_day = raw.groupby("settlement_date")["settlement_period"].max()
periods_per_day.value_counts().sort_index()

settlement_period
46      1
48    363
50      1
Name: count, dtype: int64

In [11]:
periods_per_day[periods_per_day != 48]

settlement_date
2023-03-26    46
2023-10-29    50
Name: settlement_period, dtype: int64

**Interview check:** "Why do two days have 46 and 50 periods?" Clocks go forward on
2023-03-26 (23 local hours) and back on 2023-10-29 (25 hours). Code that assumes 48
periods per day mis-stamps every reading after March.

Duplicates: exact copies and duplicate keys.

In [12]:
print("exact duplicate rows      :", raw.duplicated().sum())
print("duplicate (meter,date,period):", raw.duplicated(["meter_id", "settlement_date", "settlement_period"]).sum())
df = raw.drop_duplicates().copy()
print("rows after drop_duplicates :", len(df))

exact duplicate rows      : 40
duplicate (meter,date,period): 40
rows after drop_duplicates : 349399


Build the UTC timestamp for the whole file with the same four steps, then sort by meter and time.

In [13]:
local_midnight = pd.to_datetime(df["settlement_date"]).dt.tz_localize("Europe/London")
df["utc"] = local_midnight.dt.tz_convert("UTC") + pd.to_timedelta((df["settlement_period"] - 1) * 30, unit="min")
df = df.sort_values(["meter_id", "utc"]).reset_index(drop=True)
df.head(3)

,meter_id,settlement_date,settlement_period,kwh,utc
0,M100000,2023-01-01,1,0.734,2023-01-01 00:00:00+00:00
1,M100000,2023-01-01,2,0.638,2023-01-01 00:30:00+00:00
2,M100000,2023-01-01,3,0.394,2023-01-01 01:00:00+00:00


## 5. Finding missing periods with a full grid

Toy: one meter should have periods 1–4 but period 3 is missing. Build the complete list
of (meter, period) and `reindex` to it.

In [14]:
have = pd.DataFrame({"meter_id": ["M1", "M1", "M1"], "settlement_period": [1, 2, 4], "kwh": [0.2, 0.18, 0.16]})
have

,meter_id,settlement_period,kwh
0,M1,1,0.20
1,M1,2,0.18
2,M1,4,0.16


In [15]:
full = pd.MultiIndex.from_product([["M1"], [1, 2, 3, 4]], names=["meter_id", "settlement_period"])
on_grid = have.set_index(["meter_id", "settlement_period"])["kwh"].reindex(full)
on_grid

meter_id  settlement_period
M1        1                    0.20
          2                    0.18
          3                     NaN
          4                    0.16
Name: kwh, dtype: float64

In [16]:
on_grid[on_grid.isna()]

meter_id  settlement_period
M1        3                   NaN
Name: kwh, dtype: float64

Period 3 shows up as NaN: it is missing.

On the real file the grid is meter × date × period, with 46 / 50 on the two special days.

In [17]:
dates = pd.date_range("2023-01-01", "2023-12-31", freq="D").strftime("%Y-%m-%d")
expected = pd.Series(48, index=dates)
expected["2023-03-26"] = 46
expected["2023-10-29"] = 50
print("expected periods in the year:", expected.sum())

expected periods in the year: 17520


In [18]:
rows = []
for date, n in expected.items():
    for p in range(1, n + 1):
        rows.append((date, p))
grid = pd.DataFrame(rows, columns=["settlement_date", "settlement_period"])
meters_list = pd.DataFrame({"meter_id": sorted(df["meter_id"].unique())})
full = pd.MultiIndex.from_frame(meters_list.merge(grid, how="cross"))
print("grid rows (20 meters):", len(full))

grid rows (20 meters): 350400


In [19]:
gridded = df.set_index(["meter_id", "settlement_date", "settlement_period"])["kwh"].reindex(full)
missing_per_meter = gridded.isna().groupby("meter_id").sum()
missing_per_meter[missing_per_meter > 30]

meter_id
M100011    681
Name: kwh, dtype: int64

In [20]:
gap_days = gridded.loc["M100011"].isna().groupby("settlement_date").sum()
gap_days[gap_days > 1]        # days with more than one missing period

settlement_date
2023-04-10    48
2023-04-11    48
2023-04-12    48
2023-04-13    48
2023-04-14    48
2023-04-15    48
2023-04-16    48
2023-04-17    48
2023-04-18    48
2023-04-19    48
2023-04-20    48
2023-04-21    48
2023-04-22    48
2023-04-23    48
Name: kwh, dtype: int64

M100011 has a whole fortnight missing (48 per day from 04-10 to 04-23).

## 6. Wide vs long

Two meters, three periods. **Long**: one row per (meter, period). **Wide**: one row per
period, one column per meter.

In [21]:
long = pd.DataFrame({
    "meter": ["A", "A", "A", "B", "B", "B"],
    "period": [1, 2, 3, 1, 2, 3],
    "kwh": [0.2, 0.3, 0.1, 1.0, 1.2, 0.9],
})
long

,meter,period,kwh
0,A,1,0.2
1,A,2,0.3
2,A,3,0.1
3,B,1,1.0
4,B,2,1.2
5,B,3,0.9


In [22]:
wide = long.pivot(index="period", columns="meter", values="kwh")
wide

meter,A,B
period,,
1,0.2,1.0
2,0.3,1.2
3,0.1,0.9


Wide is what you want for cross-meter maths: the portfolio total per period is a row sum.

In [23]:
wide.sum(axis=1)

period
1    1.2
2    1.5
3    1.0
dtype: float64

`melt` goes back to long.

In [24]:
wide.reset_index().melt(id_vars="period", var_name="meter", value_name="kwh")

,period,meter,kwh
0,1,A,0.2
1,2,A,0.3
2,3,A,0.1
3,1,B,1.0
4,2,B,1.2
5,3,B,0.9


`pivot` refuses duplicate keys, which is a free check that (meter, period) is unique.

In [25]:
dup_long = pd.concat([long, long.iloc[[0]]])       # row (A, 1) twice
try:
    dup_long.pivot(index="period", columns="meter", values="kwh")
except ValueError as e:
    print("ValueError:", e)

ValueError: Index contains duplicate entries, cannot reshape


In [26]:
wide_real = df.pivot(index="utc", columns="meter_id", values="kwh")
print(wide_real.shape)
wide_real.iloc[:3, :4]

(17520, 20)


meter_id,M100000,M100001,M100002,M100003
utc,,,,
2023-01-01 00:00:00+00:00,0.734,0.108,0.280,0.190
2023-01-01 00:30:00+00:00,0.638,0.114,0.249,0.208
2023-01-01 01:00:00+00:00,0.394,0.182,0.279,0.161


## 7. The panel rule: `groupby` before `shift`

Same 6-row long frame, sorted by meter then period. A bare `shift(1)` takes the previous
*row*. At the first row of meter B, the previous row belongs to meter A.

In [27]:
long["lag1_bare"] = long["kwh"].shift(1)
long["lag1_grouped"] = long.groupby("meter")["kwh"].shift(1)
long

,meter,period,kwh,lag1_bare,lag1_grouped
0,A,1,0.2,NaN,NaN
1,A,2,0.3,0.2,0.2
2,A,3,0.1,0.3,0.3
3,B,1,1.0,0.1,NaN
4,B,2,1.2,1.0,1.0
5,B,3,0.9,1.2,1.2


Row 3 (B, period 1): the bare lag is 0.1, meter A's last value. The grouped lag is NaN,
which is right: B has no earlier period.

Same for `diff`, `pct_change`, `rolling`:

In [28]:
long["diff_grouped"] = long.groupby("meter")["kwh"].diff()
long["roll2_grouped"] = long.groupby("meter")["kwh"].rolling(2).mean().reset_index(level=0, drop=True)
long[["meter", "period", "kwh", "diff_grouped", "roll2_grouped"]]

,meter,period,kwh,diff_grouped,roll2_grouped
0,A,1,0.2,NaN,NaN
1,A,2,0.3,0.1,0.25
2,A,3,0.1,-0.2,0.20
3,B,1,1.0,NaN,NaN
4,B,2,1.2,0.2,1.10
5,B,3,0.9,-0.3,1.05


`groupby().rolling()` returns a result indexed by (meter, original row); `reset_index(level=0, drop=True)`
drops the meter level so it lines up with the frame again.

**Pitfall:** the frame must be sorted by time *within each meter* first. Shuffle it and
the grouped shift is garbage even though the groupby is there.

In [29]:
shuffled = long[["meter", "period", "kwh"]].sample(frac=1, random_state=3)
shuffled["lag1_unsorted"] = shuffled.groupby("meter")["kwh"].shift(1)
shuffled.sort_values(["meter", "period"])

,meter,period,kwh,lag1_unsorted
0,A,1,0.2,0.3
1,A,2,0.3,NaN
2,A,3,0.1,0.2
3,B,1,1.0,NaN
4,B,2,1.2,0.9
5,B,3,0.9,1.0


Compare with the `lag1_grouped` column above: the values are attached to the wrong periods.

On the real file: count the rows where a bare shift would have crossed a meter boundary.

In [30]:
df["lag1_bare"] = df["kwh"].shift(1)
df["lag1"] = df.groupby("meter_id")["kwh"].shift(1)
crossed = df["lag1_bare"].notna() & df["lag1"].isna()
print("rows whose bare shift came from another meter:", int(crossed.sum()))
df.loc[crossed, ["meter_id", "utc", "kwh", "lag1_bare", "lag1"]].head(3)

rows whose bare shift came from another meter: 19


,meter_id,utc,kwh,lag1_bare,lag1
17507,M100001,2023-01-01 00:00:00+00:00,0.108,0.364,NaN
35001,M100002,2023-01-01 00:00:00+00:00,0.280,0.129,NaN
52502,M100003,2023-01-01 00:00:00+00:00,0.190,0.228,NaN


Only 19 rows out of 349,000: a metric would barely move. That is what makes it dangerous.

## 8. `transform`: a per-meter statistic on every row

`groupby().mean()` gives one number per meter. `transform("mean")` gives that number
back on every row of the meter, so you can divide by it.

In [31]:
long = long[["meter", "period", "kwh"]].copy()
print(long.groupby("meter")["kwh"].mean())

meter
A    0.200000
B    1.033333
Name: kwh, dtype: float64


In [32]:
long["meter_mean"] = long.groupby("meter")["kwh"].transform("mean")
long["share_of_meter"] = long["kwh"] / long.groupby("meter")["kwh"].transform("sum")
long

,meter,period,kwh,meter_mean,share_of_meter
0,A,1,0.2,0.200000,0.333333
1,A,2,0.3,0.200000,0.500000
2,A,3,0.1,0.200000,0.166667
3,B,1,1.0,1.033333,0.322581
4,B,2,1.2,1.033333,0.387097
5,B,3,0.9,1.033333,0.290323


Meter A's mean 0.2 appears on all three A rows; the shares within each meter sum to 1.

## 9. Energy (sum) vs power (mean)

kWh per half hour is **energy**. Two half hours make an hour: the hourly energy is the
**sum**. The average **power** in kW is the mean kWh per half hour × 2.

In [33]:
two = pd.Series([0.20, 0.30], index=["00:00", "00:30"], name="kwh")
print(two)
print()
print("energy in the hour (kWh):", two.sum())
print("mean power (kW)         :", two.mean() * 2)
print("WRONG hourly 'kWh' via mean:", two.mean())

00:00    0.2
00:30    0.3
Name: kwh, dtype: float64

energy in the hour (kWh): 0.5
mean power (kW)         : 0.5
WRONG hourly 'kWh' via mean: 0.25


On the real file: half-hourly → hourly per meter with `pd.Grouper`.

In [34]:
hourly = df.groupby(["meter_id", pd.Grouper(key="utc", freq="h")])["kwh"].sum().reset_index()
hourly.head(3)

,meter_id,utc,kwh
0,M100000,2023-01-01 00:00:00+00:00,1.372
1,M100000,2023-01-01 01:00:00+00:00,0.763
2,M100000,2023-01-01 02:00:00+00:00,1.619


In [35]:
daily = df.groupby(["meter_id", "settlement_date"])["kwh"].sum().reset_index()
daily.head(3)

,meter_id,settlement_date,kwh
0,M100000,2023-01-01,69.969
1,M100000,2023-01-02,80.409
2,M100000,2023-01-03,82.031


## 10. Bottom-up portfolio totals and the coincidence factor

Two meters over three periods. The portfolio peak is smaller than the sum of the
individual peaks because they peak at different times.

In [36]:
w = pd.DataFrame({"A": [0.2, 0.5, 0.1], "B": [1.0, 0.4, 0.9]}, index=[1, 2, 3])
w["portfolio"] = w["A"] + w["B"]
w

,A,B,portfolio
1,0.2,1.0,1.2
2,0.5,0.4,0.9
3,0.1,0.9,1.0


In [37]:
print("A peaks at period", w["A"].idxmax(), "with", w["A"].max())
print("B peaks at period", w["B"].idxmax(), "with", w["B"].max())
print("sum of individual peaks :", w["A"].max() + w["B"].max())
print("portfolio peak          :", w["portfolio"].max())
print("coincidence factor      :", round(w["portfolio"].max() / (w["A"].max() + w["B"].max()), 2))

A peaks at period 2 with 0.5
B peaks at period 1 with 1.0
sum of individual peaks : 1.5
portfolio peak          : 1.2
coincidence factor      : 0.8


On the real file (excluding the meter-month that is in Wh instead of kWh):

In [38]:
clean = df[~((df["meter_id"] == "M100007") & (df["settlement_date"].str.startswith("2023-09")))].copy()
portfolio = clean.groupby("utc")["kwh"].sum()
individual_peaks = clean.groupby("meter_id")["kwh"].max().sum()
print("portfolio peak kWh/hh   :", round(portfolio.max(), 2), "at", portfolio.idxmax())
print("sum of individual peaks :", round(individual_peaks, 2))
print("coincidence factor      :", round(portfolio.max() / individual_peaks, 2))

portfolio peak kWh/hh   : 27.95 at 2023-11-24 17:00:00+00:00
sum of individual peaks : 51.75
coincidence factor      : 0.54


## 11. Per-meter statistics in one `agg`

Named aggregations: `new_name=("column", "function")`.

In [39]:
long[["meter", "period", "kwh"]].groupby("meter").agg(
    total_kwh=("kwh", "sum"),
    peak_kwh=("kwh", "max"),
    n_rows=("kwh", "size"),
)

,total_kwh,peak_kwh,n_rows
meter,,,
A,0.6,0.3,3
B,3.1,1.2,3


On the real file, joined with the meter attributes:

In [40]:
meters = pd.read_csv("../data/meters.csv")
stats = clean.groupby("meter_id").agg(
    annual_kwh=("kwh", "sum"),
    peak_kwh_hh=("kwh", "max"),
    n_rows=("kwh", "size"),
)
stats["peak_kw"] = stats["peak_kwh_hh"] * 2
stats["missing_share"] = 1 - stats["n_rows"] / len(grid)
stats = stats.join(meters.set_index("meter_id")[["customer_type", "annual_kwh_estimate"]])
stats["actual_vs_estimate"] = stats["annual_kwh"] / stats["annual_kwh_estimate"]
stats.round(3).sort_values("actual_vs_estimate").head(6)

,annual_kwh,peak_kwh_hh,n_rows,peak_kw,missing_share,customer_type,annual_kwh_estimate,actual_vs_estimate
meter_id,,,,,,,,
M100013,1509.694,0.964,17510,1.928,0.001,residential,2948.0,0.512
M100007,3878.004,1.212,16067,2.424,0.083,residential,3982.0,0.974
M100011,4544.086,1.664,16839,3.328,0.039,residential,4511.0,1.007
M100002,3804.000,1.367,17501,2.734,0.001,residential,3665.0,1.038
M100016,5502.002,1.653,17504,3.306,0.001,residential,5283.0,1.041
M100005,3151.860,1.019,17507,2.038,0.001,residential,3026.0,1.042


**Interview check:** "M100011 is at 0.94 of its estimate, is that customer behaviour?"
No: `missing_share` says a fortnight is missing. Read the completeness column before
interpreting the ratio.

## 12. A small panel model

One Ridge across all meters. Features per meter (lag 48 = same period yesterday,
lag 336 = same period last week), a calendar feature, and the target = same period
tomorrow. All lags inside `groupby`.

First the feature construction on the toy, so you can see what each column is.

In [41]:
mini = pd.DataFrame({
    "meter": ["A"] * 4 + ["B"] * 4,
    "t": [1, 2, 3, 4] * 2,
    "kwh": [1.0, 2.0, 3.0, 4.0, 10.0, 20.0, 30.0, 40.0],
})
mini["lag1"] = mini.groupby("meter")["kwh"].shift(1)
mini["target_next"] = mini.groupby("meter")["kwh"].shift(-1)
mini

,meter,t,kwh,lag1,target_next
0,A,1,1.0,NaN,2.0
1,A,2,2.0,1.0,3.0
2,A,3,3.0,2.0,4.0
3,A,4,4.0,3.0,NaN
4,B,1,10.0,NaN,20.0
5,B,2,20.0,10.0,30.0
6,B,3,30.0,20.0,40.0
7,B,4,40.0,30.0,NaN


`lag1` is the previous value of the same meter; `target_next` the next one. The first
row of each meter has no lag, the last no target. One `dropna` on the whole frame keeps
rows that have both.

In [42]:
mini.dropna()

,meter,t,kwh,lag1,target_next
1,A,2,2.0,1.0,3.0
2,A,3,3.0,2.0,4.0
5,B,2,20.0,10.0,30.0
6,B,3,30.0,20.0,40.0


Now the real file.

In [43]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error

pnl = clean.sort_values(["meter_id", "utc"]).copy()
g = pnl.groupby("meter_id")["kwh"]
pnl["lag0"] = pnl["kwh"]
pnl["lag48"] = g.shift(48)
pnl["lag336"] = g.shift(336)
pnl["target"] = g.shift(-48)
pnl["is_sme"] = pnl["meter_id"].map(meters.set_index("meter_id")["customer_type"]).eq("sme").astype(int)
pnl = pnl.dropna(subset=["lag48", "lag336", "target"])
pnl[["meter_id", "utc", "kwh", "lag48", "lag336", "target"]].head(3)

,meter_id,utc,kwh,lag48,lag336,target
336,M100000,2023-01-08 00:00:00+00:00,0.622,1.139,0.734,0.716
337,M100000,2023-01-08 00:30:00+00:00,0.724,0.942,0.638,0.895
338,M100000,2023-01-08 01:00:00+00:00,0.447,0.657,0.394,0.384


Chronological split at 1 October. Per-meter scaling with **train-only** mean and std.

In [44]:
split_time = pd.Timestamp("2023-10-01", tz="UTC")
train = pnl[pnl["utc"] < split_time].copy()
test = pnl[pnl["utc"] >= split_time].copy()
mu = train.groupby("meter_id")["kwh"].mean()
sd = train.groupby("meter_id")["kwh"].std()
print(mu.head(3).round(3))

meter_id
M100000    1.209
M100001    0.128
M100002    0.204
Name: kwh, dtype: float64


In [45]:
feats = ["lag0", "lag48", "lag336"]
for frame in (train, test):
    m = frame["meter_id"].map(mu)
    s = frame["meter_id"].map(sd)
    for c in feats + ["target"]:
        frame[c + "_z"] = (frame[c] - m) / s
train[["meter_id", "kwh", "lag0", "lag0_z", "target_z"]].head(3)

,meter_id,kwh,lag0,lag0_z,target_z
336,M100000,0.622,0.622,-0.549188,-0.461192
337,M100000,0.724,0.724,-0.453703,-0.293625
338,M100000,0.447,0.447,-0.713011,-0.771987


In [46]:
X_cols = [c + "_z" for c in feats] + ["is_sme"]
model = Ridge(alpha=1.0).fit(train[X_cols], train["target_z"])
test["pred"] = model.predict(test[X_cols]) * test["meter_id"].map(sd) + test["meter_id"].map(mu)
test["naive"] = test["lag0"]                # persistence: tomorrow = today
print("MAE model   :", round(mean_absolute_error(test["target"], test["pred"]), 4))
print("MAE persist.:", round(mean_absolute_error(test["target"], test["naive"]), 4))

MAE model   : 0.1372
MAE persist.: 0.1657


Always look per meter, not only pooled.

In [47]:
rows = []
for mid, gdf in test.groupby("meter_id"):
    rows.append({"meter_id": mid,
                 "mae_model": mean_absolute_error(gdf["target"], gdf["pred"]),
                 "mae_naive": mean_absolute_error(gdf["target"], gdf["naive"])})
per_meter = pd.DataFrame(rows).set_index("meter_id")
per_meter["skill"] = 1 - per_meter["mae_model"] / per_meter["mae_naive"]
per_meter = per_meter.join(meters.set_index("meter_id")["customer_type"])
per_meter.round(3).sort_values("skill").head(6)

,mae_model,mae_naive,skill,customer_type
meter_id,,,,
M100000,0.445,0.521,0.147,sme
M100015,0.764,0.901,0.151,sme
M100010,0.442,0.525,0.156,sme
M100013,0.059,0.072,0.185,residential
M100007,0.072,0.089,0.192,residential
M100016,0.097,0.120,0.193,residential


**Interview check:** "The skill is lowest for the three SMEs and higher for every
household. Why?" One pooled model learns the majority (household) behaviour; the SME
pattern (flat 08–18 on weekdays, quiet at weekends) only gets an intercept shift from
`is_sme`. Fixes: interact features with customer type, add weekday features, or fit
per type. The pooled MAE alone hides this; the per-meter table shows it.

## Checklist: settlement and panel data

| Check | Idiom |
|---|---|
| periods per day | `groupby(date).period.max().value_counts()` → 46 / 48 / 50 |
| duplicates | `duplicated()` and `duplicated([meter, date, period])` |
| gaps | full `MultiIndex` grid → `reindex` → `isna()` |
| timestamp | local midnight `tz_localize("Europe/London")` → `tz_convert("UTC")` + (period−1)·30 min |
| sort | `sort_values([meter, time])` before any shift/rolling |
| lags | `groupby(meter)[col].shift(k)`, never bare `shift` |
| rolling | `groupby().rolling()` then `reset_index(level=0, drop=True)` |
| energy vs power | kWh: sum; kW = kWh × 2 per half hour |
| resample | `groupby([meter, pd.Grouper(key=time, freq="h")]).sum()` |
| portfolio | `pivot` wide, row sums; coincidence factor |
| scaling | per-meter mean/std from train only |